In [ ]:
import langchain
import chromadb
import pypdf
import os
from IPython.display import Markdown, display

DOCS_DIR = r"../documents"
CHROMA_DIR = r"../chroma_db"

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=100)
chunks = splitter.split_documents(documents)

In [ ]:
import hashlib
def log_chunk_id(doc):
    source = doc.metadata.get("source", "")
    page = doc.metadata.get("page", "")
    h = hashlib.sha256(f"{source}-{page}-{doc.page_content}".encode()).hexdigest()
    return h

ids = [log_chunk_id(c) for c in chunks]

In [ ]:
from langchain_voyageai import VoyageAIEmbeddings
from langchain_chroma import Chroma

embedding_model = VoyageAIEmbeddings(model="voyage-4-lite")
vector_store = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embedding_model
)

In [ ]:
existing = set(vector_store.get()["ids"])
new_chunks = [c for c, i in zip(chunks, ids) if i not in existing]
new_ids = [i for i in ids if i not in existing]

if new_chunks:
    vector_store.add_documents(documents=new_chunks, ids=new_ids)
    print(f"Added {len(new_chunks)} new chunks.")
else:
    print("No new chunks to add.")

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following pieces of context to answer the question at the end.
If you don't know the answer, say that you don't know.
Reformat the answer as appropriate.
Context: {context}
Question: {question}
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "How many copies did Monster Hunter Portable 3rd sell?"

result = chain.invoke(question)
display(Markdown(result))

In [ ]:
import os
import sys
from pathlib import Path

if Path("rag").is_dir() is False:
    os.chdir("..")
sys.path.insert(0, ".")

from dotenv import load_dotenv

load_dotenv()

from rag.chunking import chunk_documents
from rag.config import RERANK_K, RETRIEVAL_K
from rag.ingest import load_documents
from rag.vectorstore import (
    get_compression_retriever,
    get_reranker,
    get_retriever,
    get_vector_store,
    upsert_chunks,
)

documents = load_documents()
print(f"Loaded {len(documents)} documents.")

chunks = chunk_documents(documents)
print(f"split documents in {len(chunks)} chunks.")

vector_store = get_vector_store()
added = upsert_chunks(vector_store, chunks)
print(f"Added {added} new chunks." if added else "No new chunks to add.")

retriever = get_retriever(vector_store)
reranker = get_reranker()
compression_retriever = get_compression_retriever(reranker, retriever)

question = "Which Nintendo consoles have had Monster Hunter games?"

reranked = compression_retriever.invoke(question)
scored = vector_store.similarity_search_with_score(question, k=RETRIEVAL_K)

print(f"\nQ: {question}")

print(f"\nBase retriever top {RERANK_K} (distance, lower = closer):")
for i, (d, score) in enumerate(scored[:RERANK_K], 1):
    print(f"  {i}. {d.metadata.get('source')}   distance={score:.4f}")
    # print(f"     {d.page_content[:250].strip()}")

print(f"\nReranked top {RERANK_K}:")
for i, d in enumerate(reranked, 1):
    score = d.metadata.get("relevance_score")
    score = f"{score:.4f}"
    print(f"{i}. {d.metadata.get('source')}   score={score}")
    print(f"     {d.page_content[:100].strip()}")